In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import pandas as pd
import numpy as np

train = pd.read_pickle('data/train_processed.pkl')
test = pd.read_pickle('data/test_processed.pkl')

Mounted at /content/drive


In [ ]:
# ============================================================
# MLflow / DagsHub experiment tracking
# ============================================================
!pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

import mlflow

EXPERIMENT_NAME = 'XGBoost_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)

# Close any run left open by a previous (interrupted) execution of this notebook.
if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

# Cell 2 - WMAE metric


In [2]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Cell 3 - Chronological validation split (same cutoff as LightGBM notebook, for fair comparison)


In [3]:
train = train.sort_values('Date')
val = train[train['Date'] >= train['Date'].max() - pd.Timedelta(weeks=39)].copy()
fit = train[train['Date'] < val['Date'].min()].copy()
print('fit ends:', fit['Date'].max())
print('val range:', val['Date'].min(), 'to', val['Date'].max())

fit ends: 2012-01-20 00:00:00
val range: 2012-01-27 00:00:00 to 2012-10-26 00:00:00


# Cell 4 - Baseline (identical to LightGBM notebook - same yardstick)


In [4]:
baseline_pred = val['lag_52'].fillna(val['storedept_median_sales'])
baseline_score = wmae(val['Weekly_Sales'], baseline_pred, val['IsHoliday'])
print('52-week seasonal baseline WMAE:', baseline_score)

52-week seasonal baseline WMAE: 1789.9133525504185


# Cell 5 - Prepare features for XGBoost


In [5]:
# XGBoost needs categorical columns cast to 'category' dtype AND enable_categorical=True.
cat_cols = ['Store','Dept','Type','holiday_type']
for df in [fit, val, test]:
    for c in cat_cols:
        df[c] = df[c].astype('category')

drop_cols = ['Weekly_Sales','Date','Id'] if 'Id' in fit.columns else ['Weekly_Sales','Date']
feature_cols = [c for c in fit.columns if c not in drop_cols]

X_fit, y_fit = fit[feature_cols], fit['Weekly_Sales']
X_val, y_val = val[feature_cols], val['Weekly_Sales']
print('Total features going in:', len(feature_cols))

Total features going in: 52


# Cell 6 - Train XGBoost on all features (this run informs feature selection)


In [7]:
import xgboost as xgb

model_all = xgb.XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=8,
    device='cuda',
    enable_categorical=True,
    early_stopping_rounds=50,
    random_state=42,
    eval_metric='mae'
)
model_all.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    verbose=100
)

[0]	validation_0-mae:12978.03028
[100]	validation_0-mae:1937.09909
[200]	validation_0-mae:1353.56596
[300]	validation_0-mae:1322.30452
[400]	validation_0-mae:1304.95548
[500]	validation_0-mae:1293.78379
[600]	validation_0-mae:1284.66499
[700]	validation_0-mae:1275.24414
[800]	validation_0-mae:1270.60796
[900]	validation_0-mae:1267.84729
[1000]	validation_0-mae:1265.59615
[1100]	validation_0-mae:1262.35746
[1200]	validation_0-mae:1260.57425
[1300]	validation_0-mae:1258.92722
[1400]	validation_0-mae:1258.74586
[1426]	validation_0-mae:1258.80333


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device='cuda', early_stopping_rounds=50,
             enable_categorical=True, eval_metric='mae', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=None, num_parallel_tree=None, ...)

# Cell 7 - Feature importance -> single-pass feature selection


In [8]:
# Same reasoning as LightGBM: trees don't strictly need this, but we prune
# zero-importance and redundant correlated features (overlapping lags) for
# a cleaner, faster, more interpretable final model. No comparison run.
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_all.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

selected_features = importances[importances['importance'] > 0]['feature'].tolist()
print(f'Dropped {len(feature_cols) - len(selected_features)} zero-importance features')
print('Final feature count:', len(selected_features))

                    feature  importance
30                    lag_1    0.511174
51   storedept_median_sales    0.265708
36                   lag_52    0.015734
24             holiday_type    0.014720
27             week_of_year    0.013178
25                     year    0.011721
1                      Dept    0.011538
26                    month    0.011001
38         rolling_mean_4_x    0.010457
31                    lag_2    0.008459
29                 week_cos    0.008417
2                 IsHoliday    0.007191
28                 week_sin    0.007026
0                     Store    0.006792
32                    lag_4    0.006676
46           sales_change_1    0.005672
41        rolling_mean_26_x    0.005200
39       rolling_median_8_x    0.004987
7                 MarkDown3    0.004609
40        rolling_mean_13_x    0.004576
47     sales_per_store_size    0.004555
35                   lag_51    0.004329
20           markdown_count    0.004319
37                   lag_53    0.004290


# Cell 8 - Retrain ONCE on selected features - this is our final model


In [10]:
X_fit_sel = fit[selected_features]
X_val_sel = val[selected_features]

model = xgb.XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=8,
    device='cuda',
    enable_categorical=True,
    early_stopping_rounds=50,
    random_state=42,
    eval_metric='mae'
)
model.fit(
    X_fit_sel, y_fit,
    eval_set=[(X_val_sel, y_val)],
    verbose=100
)

val_pred = model.predict(X_val_sel)
wmae_final = wmae(y_val, val_pred, val['IsHoliday'])
print('Baseline:        ', baseline_score)
print('Final XGBoost WMAE:', wmae_final)

[0]	validation_0-mae:12978.02868
[100]	validation_0-mae:1935.66814
[200]	validation_0-mae:1359.39114
[300]	validation_0-mae:1323.69992
[400]	validation_0-mae:1305.30319
[500]	validation_0-mae:1292.02259
[600]	validation_0-mae:1283.01405
[700]	validation_0-mae:1273.68478
[800]	validation_0-mae:1267.57848
[900]	validation_0-mae:1265.21051
[1000]	validation_0-mae:1261.29890
[1100]	validation_0-mae:1257.38556
[1200]	validation_0-mae:1255.78326
[1300]	validation_0-mae:1255.12437
[1400]	validation_0-mae:1254.44028
[1463]	validation_0-mae:1257.18341


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [22:26:53] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Baseline:         1789.9133525504185
Final XGBoost WMAE: 1277.603208859651


# Cell 9 - Holiday vs non-holiday breakdown


In [11]:
is_hol = val['IsHoliday'].values
print('Holiday WMAE:    ', wmae(y_val[is_hol], val_pred[is_hol], val['IsHoliday'][is_hol]))
print('Non-holiday WMAE:', wmae(y_val[~is_hol], val_pred[~is_hol], val['IsHoliday'][~is_hol]))

Holiday WMAE:     1393.6646832988338
Non-holiday WMAE: 1246.843665364381


# Cell 10 - Save model locally


In [13]:
import os
import joblib

joblib.dump(model, 'models/xgboost_best.pkl')
print('Saved.')

Saved.


---
# Production pipeline

The requirement is that the saved pipeline runs on the **raw, un-preprocessed test
set**. The model above was trained on `data/*_processed.pkl`, so the whole of
`feature_engineering.ipynb` has to be reachable from inside the pipeline object.

`WalmartFeatureEngineer` below is that notebook re-expressed as a scikit-learn
transformer. It changes nothing about the features: it learns the train history,
rolling tails and median aggregates in `fit`, and rebuilds the same 53 columns in
`transform`. The final pipeline is

    raw rows -> WalmartFeatureEngineer -> CategoricalCaster -> FeatureSelector -> model

In [ ]:
# ============================================================
# Preprocessing pipeline components
# ============================================================
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
LAG_WEEKS = [1, 2, 4, 13, 26, 51, 52, 53]
ROLL_COLS = ['rolling_mean_4', 'rolling_median_8', 'rolling_mean_13', 'rolling_mean_26']

SUPERBOWL = ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']
LABORDAY = ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']
THANKSGIVING = ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']
CHRISTMAS = ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']


def holiday_type_of(date):
    d = date.strftime('%Y-%m-%d')
    if d in SUPERBOWL:
        return 'SuperBowl'
    if d in LABORDAY:
        return 'LaborDay'
    if d in THANKSGIVING:
        return 'Thanksgiving'
    if d in CHRISTMAS:
        return 'Christmas'
    return 'Normal'


class WalmartFeatureEngineer(BaseEstimator, TransformerMixin):
    """feature_engineering.ipynb, expressed as a transformer.

    fit(X, y)   X = raw train rows (Store, Dept, Date, IsHoliday), y = Weekly_Sales.
                Learns the sales history, the rolling-stat tail per Store-Dept and
                the median aggregates - all from train only.
    transform(X) X = raw rows (the Kaggle test set). Rebuilds the feature frame the
                model was trained on. Lags and rolling stats can only look back into
                the train history, exactly as the original notebook does for test.
    """

    def __init__(self, features, stores, lag_weeks=tuple(LAG_WEEKS)):
        self.features = features
        self.stores = stores
        self.lag_weeks = lag_weeks

    def fit(self, X, y=None):
        raw = X.copy()
        raw['Date'] = pd.to_datetime(raw['Date'])
        if y is not None:
            raw['Weekly_Sales'] = np.asarray(y)
        raw = raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        self.history_ = raw[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
        self.train_keys_ = raw[['Store', 'Date']].drop_duplicates()

        rs = self.history_.copy()
        g = rs.groupby(['Store', 'Dept'])['Weekly_Sales']
        rs['rolling_mean_4'] = g.shift(1).rolling(4).mean().reset_index(drop=True)
        rs['rolling_median_8'] = g.shift(1).rolling(8).median().reset_index(drop=True)
        rs['rolling_mean_13'] = g.shift(1).rolling(13).mean().reset_index(drop=True)
        rs['rolling_mean_26'] = g.shift(1).rolling(26).mean().reset_index(drop=True)

        # test is in the future, so every test row gets the last rolling value
        # available for its Store-Dept
        self.last_roll_ = (rs.sort_values('Date')
                             .groupby(['Store', 'Dept'])[ROLL_COLS]
                             .last().reset_index())

        self.dept_median_ = raw.groupby('Dept')['Weekly_Sales'].median().rename('dept_median_sales')
        self.store_median_ = raw.groupby('Store')['Weekly_Sales'].median().rename('store_median_sales')
        self.storedept_median_ = (raw.groupby(['Store', 'Dept'])['Weekly_Sales']
                                     .median().rename('storedept_median_sales'))
        return self

    def _macro(self, test_keys):
        # CPI / Unemployment are forward-filled per Store across train + test, so a
        # test week inherits the last value observed during training.
        combined = pd.concat([self.train_keys_, test_keys], ignore_index=True).drop_duplicates()
        combined = combined.merge(self.features[['Store', 'Date', 'CPI', 'Unemployment']],
                                  on=['Store', 'Date'], how='left')
        combined = combined.sort_values(['Store', 'Date'])
        for col in ['CPI', 'Unemployment']:
            combined[f'{col}_missing'] = combined[col].isna().astype('int8')
            combined[col] = combined.groupby('Store')[col].ffill()
        return combined

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        df = (df.merge(self.features, on=['Store', 'Date', 'IsHoliday'],
                       how='left', validate='many_to_one')
                .merge(self.stores, on='Store', how='left', validate='many_to_one'))

        for col in MARKDOWN_COLS:
            df[f'{col}_missing'] = df[col].isna().astype('int8')
            df[col] = df[col].fillna(0)
        df['markdown_total'] = df[MARKDOWN_COLS].sum(axis=1)
        df['markdown_count'] = (df[MARKDOWN_COLS] > 0).sum(axis=1)
        df['has_any_markdown'] = (df['markdown_total'] > 0).astype('int8')

        macro = self._macro(df[['Store', 'Date']].drop_duplicates())
        df = df.drop(columns=['CPI', 'Unemployment']).merge(macro, on=['Store', 'Date'], how='left')

        df['holiday_type'] = df['Date'].apply(holiday_type_of)

        df['year'] = df['Date'].dt.year
        df['month'] = df['Date'].dt.month
        df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
        df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
        df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)

        for w in self.lag_weeks:
            shifted = self.history_.copy()
            shifted['Date'] = shifted['Date'] + pd.Timedelta(weeks=w)
            shifted = shifted.rename(columns={'Weekly_Sales': f'lag_{w}'})
            df = df.merge(shifted, on=['Store', 'Dept', 'Date'], how='left')

        df = df.merge(self.last_roll_, on=['Store', 'Dept'], how='left')
        # A clean run of feature_engineering.ipynb produces plain rolling column
        # names. Re-running its merge cell in the same kernel produces _x / _y
        # duplicates instead. Emit both spellings so the pipeline matches either
        # schema; FeatureSelector keeps only the columns the model was trained on.
        for c in ROLL_COLS:
            df[f'{c}_x'] = df[c]
            df[f'{c}_y'] = df[c]

        df['sales_change_1'] = df['lag_1'] - df['lag_2']
        df['sales_per_store_size'] = df['lag_52'] / df['Size']
        df['markdown_per_store_size'] = df['markdown_total'] / df['Size']

        df = (df.merge(self.dept_median_, on='Dept', how='left')
                .merge(self.store_median_, on='Store', how='left')
                .merge(self.storedept_median_, on=['Store', 'Dept'], how='left'))
        return df


class CategoricalCaster(BaseEstimator, TransformerMixin):
    """Cast the categorical columns to pandas 'category' dtype."""

    def __init__(self, cat_cols):
        self.cat_cols = cat_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cat_cols:
            if c in X.columns:
                X[c] = X[c].astype('category')
        return X


class FeatureSelector(BaseEstimator, TransformerMixin):
    """Keep the model's features, in the order it was trained on."""

    def __init__(self, features):
        self.features = features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.features]


def build_full_pipeline(fitted_model, model_features, feature_engineer):
    # Every transformer is stateless or already fitted, so nothing is retrained here.
    return Pipeline([
        ('feature_engineering', feature_engineer),
        ('cast_categoricals', CategoricalCaster(cat_cols)),
        ('select_features', FeatureSelector(model_features)),
        ('model', fitted_model),
    ])

In [ ]:
# ============================================================
# Fit the feature engineer on the RAW training data
# ============================================================
train_raw = pd.read_csv('data/train.csv', parse_dates=['Date'])
test_raw = pd.read_csv('data/test.csv', parse_dates=['Date'])
features_raw = pd.read_csv('data/features.csv', parse_dates=['Date'])
stores_raw = pd.read_csv('data/stores.csv')

feature_engineer = WalmartFeatureEngineer(features_raw, stores_raw).fit(
    train_raw.drop(columns='Weekly_Sales'),
    train_raw['Weekly_Sales'],
)
print('Feature engineer fitted on', len(train_raw), 'raw training rows.')

In [ ]:
# ============================================================
# The final XGBoost model, wrapped as a raw-input pipeline
# ============================================================
final_pipeline = build_full_pipeline(model, selected_features, feature_engineer)
print('Final model:', len(selected_features), 'features')

In [ ]:
# ============================================================
# Sanity check: raw -> pipeline must reproduce the processed-data predictions
# ============================================================
_ref = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
_ref_pred = model.predict(_ref[selected_features])
_pipe_pred = final_pipeline.predict(test_raw)

pipeline_max_abs_diff = float(np.abs(_ref_pred - _pipe_pred).max())
print('max |pipeline(raw) - model(processed)| =', pipeline_max_abs_diff)
assert pipeline_max_abs_diff < 1e-6, 'pipeline does not reproduce the trained model'

In [ ]:
# ============================================================
# MLflow - one experiment per architecture, one run per stage
# ============================================================
import os
import tempfile

import mlflow.sklearn

PREPROCESSING_STEPS = [
    {'step': 'load_raw', 'detail': 'read train/test/features/stores, parse Date, sort by Store-Dept-Date'},
    {'step': 'merge', 'detail': "features on ['Store','Date','IsHoliday'] + stores on ['Store']"},
    {'step': 'markdown_missing_indicators',
     'detail': 'MarkDown*_missing flags, fillna(0), markdown_total, markdown_count, has_any_markdown'},
    {'step': 'macro_forward_fill', 'detail': 'per-Store ffill of CPI / Unemployment across train+test'},
    {'step': 'holiday_type', 'detail': 'SuperBowl / LaborDay / Thanksgiving / Christmas / Normal'},
    {'step': 'calendar_features', 'detail': 'year, month, week_of_year, week_sin, week_cos'},
    {'step': 'exact_date_lags', 'detail': 'date-shifted merges; test looks back into train history only'},
    {'step': 'rolling_statistics', 'detail': 'mean_4 / median_8 / mean_13 / mean_26; test gets the last train value'},
    {'step': 'trend_and_size_features', 'detail': 'sales_change_1, sales_per_store_size, markdown_per_store_size'},
    {'step': 'fallback_aggregates', 'detail': 'dept / store / store-dept median sales, from train only'},
]




# ---------- run 1: cleaning / preprocessing ----------
with mlflow.start_run(run_name='XGBoost_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'markdown_cols': MARKDOWN_COLS,
        'lag_weeks': LAG_WEEKS,
        'rolling_windows': [4, 8, 13, 26],
        'forward_filled_cols': ['CPI', 'Unemployment'],
        'merge_validate': 'many_to_one',
        'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })
    mlflow.log_metrics({
        'train_rows': train.shape[0],
        'train_cols': train.shape[1],
        'test_rows': test.shape[0],
        'test_cols': test.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()),
        'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store', 'Dept']].drop_duplicates().shape[0]),
    })
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )

# ---------- run 2: baseline ----------
with mlflow.start_run(run_name='XGBoost_Baseline'):
    mlflow.set_tag('stage', 'baseline')
    mlflow.log_param('strategy', 'lag_52, falling back to storedept_median_sales')
    mlflow.log_metric('val_wmae', baseline_score)

# ---------- run 3: feature selection ----------
with mlflow.start_run(run_name='XGBoost_Feature_Selection'):
    mlflow.set_tag('stage', 'feature_selection')
    mlflow.log_params(model_all.get_params())
    mlflow.log_params({
        'selection_rule': 'importance > 0 on the all-features model',
        'n_features_before': len(feature_cols),
        'n_features_after': len(selected_features),
    })
    mlflow.log_metrics({
        'val_mae_all_features': float(model_all.best_score),
        'n_features_dropped': len(feature_cols) - len(selected_features),
        'best_iteration': int(model_all.best_iteration),
    })
    mlflow.log_dict({'features': feature_cols}, 'features/all_features.json')
    mlflow.log_dict({'features': selected_features}, 'features/selected_features.json')
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'feature_importances.csv')
        importances.to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='features')

# ---------- run 4: validation (chronological holdout) ----------
with mlflow.start_run(run_name='XGBoost_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.log_params({
        'scheme': 'chronological holdout (no k-fold: the target is a time series)',
        'val_weeks': 39,
        'fit_end': str(fit['Date'].max().date()),
        'val_start': str(val['Date'].min().date()),
        'val_end': str(val['Date'].max().date()),
        'fit_rows': len(fit),
        'val_rows': len(val),
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_score,
        'final_wmae': wmae_final,
        'val_wmae_holiday': wmae(y_val[is_hol], val_pred[is_hol], val['IsHoliday'][is_hol]),
        'val_wmae_non_holiday': wmae(y_val[~is_hol], val_pred[~is_hol], val['IsHoliday'][~is_hol]),
        'improvement_over_baseline': baseline_score - wmae_final,
    })

# ---------- run 5: final model + full raw-input pipeline ----------
with mlflow.start_run(run_name='XGBoost_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'xgboost'})
    mlflow.log_params(model.get_params())
    mlflow.log_param('n_features', len(selected_features))
    mlflow.log_metrics({
        'final_wmae': wmae_final,
        'baseline_wmae': baseline_score,
        'pipeline_vs_processed_max_abs_diff': pipeline_max_abs_diff,
    })

    model_info = mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path='pipeline',
        # cloudpickle embeds the notebook-defined transformers in the artifact;
        # skops (the new default) refuses to serialise __main__ classes
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        input_example=test_raw.head(5),
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    mlflow.log_artifact('models/xgboost_best.pkl', artifact_path='estimator')
    print('XGBoost final run:', final_run.info.run_id)